# Připojení Google drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs("/content/drive/MyDrive/4IZ175", exist_ok=True)

# Import knihoven

In [ ]:
#!pip install tensorflow tensorflow-datasets

In [ ]:
import tensorflow_datasets as tfds
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow.keras import layers
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Načtení datasetu


In [ ]:
(ds_train,ds_val, ds_test), ds_info = tfds.load(
    'cifar10',
    split=['train[:40000]', 'train[40000:50000]', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
)

In [ ]:
# nastaveni seedu na stabilni chovani
keras.utils.set_random_seed(42)

# Základní preprocessing




In [ ]:
CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
AUTOTUNE = tf.data.AUTOTUNE
IMG_SIZE = 32
batch_size = 32

def resize_and_rescale(image, label):
  image = tf.cast(image, tf.float32)
  image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
  return image, label

ds_train = (
    ds_train
    .shuffle(2000)
    .map(resize_and_rescale, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

ds_val = (
    ds_val
    .map(resize_and_rescale, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

ds_test= (
    ds_test
    .map(resize_and_rescale, num_parallel_calls=AUTOTUNE)
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

Základní první model


In [ ]:
inputs = keras.Input(shape=(32, 32, 3))

x = layers.Rescaling(1./255)(inputs)


x = layers.Conv2D(32, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=2)(x)


x = layers.Conv2D(64, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=2)(x)


x = layers.Conv2D(128, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=2)(x)


x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation="softmax", name="output")(x)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

Funkce k získání nejlepší epochy, funcke vybere epochu s nejvyšší hodnotou validační přesnosti z historie trénování

In [ ]:
def get_best_epoch(history):
    best_epoch = np.argmax(history.history['val_accuracy'])
    best_train_acc = history.history['accuracy'][best_epoch]
    best_val_acc = history.history['val_accuracy'][best_epoch]
    return best_epoch, best_train_acc, best_val_acc

In [ ]:
callback = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=20, min_delta = 0.002, restore_best_weights=True)
history = model.fit(ds_train, epochs=80, validation_data=ds_val, callbacks=[callback])

best_epoch, best_train_acc, best_val_acc  = get_best_epoch(history)
print(f'Best epoch: {best_epoch}')
print(f'Best train accuracy: {best_train_acc}')
print(f'Best validation accuracy: {best_val_acc}')

Funkce k vykreslení trénovací a validační přesnost modelu v průběhu trénování

In [ ]:
import matplotlib.pyplot as plt
def plot_history(history):
  accuracy = history.history["accuracy"]
  val_accuracy = history.history["val_accuracy"]
  loss = history.history["loss"]
  val_loss = history.history["val_loss"]
  epochs = range(1, len(accuracy) + 1)
  plt.plot(epochs, accuracy, "bo", label="Training accuracy")
  plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
  plt.title("Training and validation accuracy")
  plt.legend()
  plt.figure()
  plt.plot(epochs, loss, "bo", label="Training loss")
  plt.plot(epochs, val_loss, "b", label="Validation loss")
  plt.title("Training and validation loss")
  plt.legend()
  plt.show()

In [ ]:
plot_history(history)

S trénovací přesností okolo 90% a validační oscilující okolo 75% model pravděpodobně overfituje, jako prní krok na zlepšení modelu jsme zvolili augmentaci dat

Nová vrstva, která provede základní augmentaci dat.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")


Vytvoření modelu s augmentační vrstvou na trénování, architektura jinak nezměněná


In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)


x = layers.Conv2D(32, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=2)(x)


x = layers.Conv2D(64, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=2)(x)


x = layers.Conv2D(128, kernel_size=3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(pool_size=2)(x)


x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation="softmax", name="output")(x)

model = keras.Model(inputs=inputs, outputs=outputs, name="CNN_32x32")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callback = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=20, min_delta = 0.002, restore_best_weights=True)
history = model.fit(ds_train, epochs=80, validation_data=ds_val, callbacks=[callback])

best_epoch, best_train_acc, best_val_acc  = get_best_epoch(history)
print(f'Best epoch: {best_epoch}')
print(f'Best train accuracy: {best_train_acc}')
print(f'Best validation accuracy: {best_val_acc}')

In [ ]:
plot_history(history)

Augmentace dat se zbavila problému s overfittingem, trénovací a validační přesnostijsou velmi podobné, nicméně validační přesnost je stále okolo 75%, model tedy na složitějších augmentovaných datech underfituje


Jako další pokus o zlepšení bylo tedy vybráno ponechat augmentační vrstvu a probloubit model

In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

# zvetseni poctu filtru z 32 na 64
x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.25)(x)

# zvetseni z 64 na 128
x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.25)(x)

# zvetseni poctu filtru z 128 na 256
x = layers.Conv2D(256, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.25)(x)

# pridani ctvrteho konvolucniho bloku
x = layers.Conv2D(512, 3, padding="same", activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.25)(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation="softmax")(x)

In [ ]:
model = keras.Model(inputs=inputs, outputs=outputs, name="CNN_32x32")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callback = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=20, min_delta = 0.002, restore_best_weights=True)
history = model.fit(ds_train, epochs=80, validation_data=ds_val, callbacks=[callback])



In [ ]:
best_epoch, best_train_acc, best_val_acc  = get_best_epoch(history)
print(f'Best epoch: {best_epoch}')
print(f'Best train accuracy: {best_train_acc}')
print(f'Best validation accuracy: {best_val_acc}')

In [ ]:
plot_history(history)

Hlubší model s augmentací dat dosahuje značně lepších výsledků, neoverfituje ani underfituje a má očekávaně vyšší přesnost na validační sadě.

In [ ]:
total_steps = len(ds_train) * 80

lr_schedule = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=1e-3,
    decay_steps=total_steps,
    alpha=1e-6
)


Další pokud o zlepšení je cosínový learning rate schedule a label smoothing

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

Vytvareni one_hot encodingu pro categorical cross entropy


In [ ]:
def one_hot(image, label):
    label = tf.one_hot(label, depth=10)
    return image, label
ds_train_one_hot = (
    ds_train
    .map(one_hot)
)

ds_val_one_hot = (
    ds_val
    .map(one_hot)
)

ds_test_one_hot= (
    ds_test
    .map(one_hot)
)

In [ ]:
callback = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=20, min_delta = 0.002, restore_best_weights=True)
history = model.fit(ds_train_one_hot, epochs=80, validation_data=ds_val_one_hot, callbacks=[callback])

best_epoch, best_train_acc, best_val_acc  = get_best_epoch(history)
print(f'Best epoch: {best_epoch}')
print(f'Best train accuracy: {best_train_acc}')
print(f'Best validation accuracy: {best_val_acc}')

In [ ]:
plot_history(history)

Horší než minulý model

# Hyperparameter tuning


Vycházíme z hlubšího modelu s augmentací dat ale bez label smoothingu

In [ ]:
%pip install optuna optuna-integration

Na optimalizaci hyperparametrů jsme zvolili knihovnu optuna, která využívá propabilistických modelů na optimalizace a je ze zkušeností značně rychlejší než grid search

In [ ]:
import optuna
from optuna.integration import TFKerasPruningCallback

In [ ]:
EPOCHS_PER_TRIAL = 15
N_TRIALS         = 30
PATIENCE         = 4
NUM_CLASSES      = 10
INPUT_SHAPE      = (32, 32, 3)

Funkce na vytvoření modelu pro optimalizaci. U numerických hyperparametrů nastavujeme meze a u ostatních možnosti na výběr. Funkce vytváří modely vycházející z předposledního modelu.

In [ ]:
def build_model(trial):
    base_filters   = trial.suggest_categorical("base_filters", [32, 64])
    activation     = trial.suggest_categorical("activation", ["relu", "elu", "swish"])
    use_bn         = trial.suggest_categorical("batch_norm", [True, False])
    conv_dropout   = trial.suggest_float("conv_dropout", 0.1, 0.4, step=0.1)
    dense_units    = trial.suggest_categorical("dense_units", [128, 256, 512])
    dense_dropout  = trial.suggest_float("dense_dropout", 0.3, 0.6, step=0.1)
    learning_rate  = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["adam", "adamw"])


    inputs = keras.Input(shape=INPUT_SHAPE)
    x = data_augmentation(inputs)
    x = layers.Rescaling(1.0 / 255)(x)

    filter_schedule = [
        base_filters,
        base_filters * 2,
        base_filters * 4,
        base_filters * 8,
    ]

    for i, filters in enumerate(filter_schedule):
        x = layers.Conv2D(filters, 3, padding="same", activation=activation)(x)
        if use_bn:
            x = layers.BatchNormalization()(x)
        if i < 3:
            x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(conv_dropout)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(dense_units, activation=activation)(x)
    x = layers.Dropout(dense_dropout)(x)
    outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

    model = keras.Model(inputs, outputs)

    if optimizer_name == "adamw":
        weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)
        optimizer = keras.optimizers.AdamW(learning_rate=learning_rate,
                                           weight_decay=weight_decay)
    else:
        optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

Optimalizace pomocí optuny potřebuje objektivní funkci k maximalizaci/minimalizaci. Vybíráme maximální přenost na validační sadě.

In [ ]:
def objective(trial):
    model = build_model(trial)

    callbacks = [
        TFKerasPruningCallback(trial, monitor="val_accuracy"),
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy",
            patience=PATIENCE,
            restore_best_weights=True,
        ),
    ]

    history = model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=EPOCHS_PER_TRIAL,
        callbacks=callbacks,
        verbose=0,
    )

    return max(history.history["val_accuracy"])

Funkce na spuštění samotné optimalizace. Nastavujeme seed a toleranci pěti prvních epoch než je využit early stopping na modely s validační přesností pod mediánem předchozích modelů.

In [ ]:
def run_study():

    sampler = optuna.samplers.TPESampler(seed=42)
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=5,
        n_warmup_steps=5,
    )

    study = optuna.create_study(
        study_name="cifar10_cnn_study",
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        load_if_exists=True,
    )

    study.optimize(
        objective,
        n_trials=N_TRIALS,
        show_progress_bar=True,
    )

    return study

Vytrénování nejlepšího nalezeného modelu.

In [ ]:
def retrain_best(study, epochs = 80):

    model = build_model(study.best_trial)

    model.fit(
        ds_train,
        validation_data=ds_val,
        epochs=epochs,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor="val_accuracy",
                                          patience=8,
                                          restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor="val_loss",
                                              factor=0.5,
                                              patience=4),
        ],
        verbose=1,
    )
    return model

In [ ]:
study = run_study()

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
best_model = retrain_best(study)

# Výsledky optimalizace


Použitím optuna tuneru jsme byli schopni dosáhnout modelů s vyšší validační přesností.

Z prvního běhu Optuna hyperparameter tuningu, ze kterého zde bohužel nemáme výstup, vzešel lepší model než z druhého běhu, rozhodli jsme se tedy manuálně vytvořit ten. Model má tyto hyperparametry: aktivace ELU, 4 konvoluční bloky (64→128→256→512 filtrů), Batch Normalization, conv dropout 0.2, dense vrstva 512 neuronů, dense dropout 0.4, optimizer Adam s learning rate 0.00109. Model byl natrénován na 80 epoch s early stopping (patience 20), nejlepší epocha byla 43. s validační accuracy 83.76 %.


In [ ]:
inputs = keras.Input(shape=(32, 32, 3))
x = data_augmentation(inputs)
x = layers.Rescaling(1./255)(x)

# zvetseni poctu filtru z 32 na 64
x = layers.Conv2D(64, 3, padding="same", activation="elu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.2)(x)

# zvetseni z 64 na 128
x = layers.Conv2D(128, 3, padding="same", activation="elu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.2)(x)

# zvetseni poctu filtru z 128 na 256
x = layers.Conv2D(256, 3, padding="same", activation="elu")(x)
x = layers.BatchNormalization()(x)
x = layers.MaxPooling2D(2)(x)
x = layers.Dropout(0.2)(x)

# pridani ctvrteho konvolucniho bloku
x = layers.Conv2D(512, 3, padding="same", activation="elu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.2)(x)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(512, activation="elu")(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(10, activation="softmax")(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="CNN_32x32")

In [ ]:

optimizer = keras.optimizers.Adam(learning_rate=0.0010864190323862648)

model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

In [ ]:
callback = keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=20, min_delta = 0.002, restore_best_weights=True)
history = model.fit(ds_train, epochs=50, validation_data=ds_val, callbacks=[callback])

best_epoch, best_train_acc, best_val_acc  = get_best_epoch(history)
print(f'Best epoch: {best_epoch}')
print(f'Best train accuracy: {best_train_acc}')
print(f'Best validation accuracy: {best_val_acc}')

In [ ]:
plot_history(history)

In [ ]:
import datetime

log_dir = "/content/drive/MyDrive/4IZ175/logs"
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=log_dir,
    histogram_freq=0,
    write_graph=False
)

In [ ]:
# eval na test datech
test_loss, test_acc = model.evaluate(ds_test)
print(f"\n>>> Testovací accuracy: {test_acc:.4f} <<<")
print(f">>> Testovací loss:     {test_loss:.4f} <<<")

# CONFUSION MATRIX + CLASSIFICATION REPORT
y_pred = []
y_true = []
for images, labels in ds_test:
    preds = model.predict(images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

y_pred = np.array(y_pred)
y_true = np.array(y_true)

print("Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

# Confusion matrix – heatmapa
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predikovaná třída')
plt.ylabel('Skutečná třída')
plt.title(f'Confusion Matrix (test acc = {test_acc:.4f})')
plt.tight_layout()
plt.show()


# per class acc barplot
per_class_acc = cm.diagonal() / cm.sum(axis=1)
plt.figure(figsize=(10, 5))
bars = plt.bar(CLASS_NAMES, per_class_acc, color='steelblue')
plt.axhline(y=test_acc, color='red', linestyle='--', label=f'Průměr: {test_acc:.4f}')
plt.ylabel('Accuracy')
plt.title('Accuracy po třídách')
plt.xticks(rotation=45)
plt.legend()
for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

# Interpretace výsledků

Z heatmapy a  porovnání  přesnosti v rámci tříd můžeme vidět sníženou  přesnost a miskvalifikaci v rámci třír "cat", "dog" a "deer" kde dochází hlavně k záměně mezi kočkami a psy. Toto je celkem logické vzhledem k tomu že na obrázku 32x32 pixelů mohou vzhledem k proporcím vypadat zvířata stejně aspoň pro model.

## ulozeni modelu

In [ ]:
os.makedirs("/content/drive/MyDrive/4IZ175", exist_ok=True)

save_path = f"/content/drive/MyDrive/4IZ175/misun-vojtech-obraz.keras"
model.save(save_path)